# PowerOps_v1 — Notebook 01: Data Loading & Exploration

This notebook is the first step in building **PowerOps**, a Retrieval-Augmented
Generation (RAG) assistant for DevOps issue management.

**Goal of this notebook:** load the real DevOps export (`data/devops_mgmt_data.json`),
understand its actual schema, validate it, normalize it into a clean internal
representation, and profile it (counts, distributions, missing data, duplicates).

We deliberately do **not** touch LangChain, embeddings, or Pinecone here — this
notebook is purely about understanding and preparing the data. RAG wiring starts
in Notebook 02.

### Important: this is real exported data, not a synthetic template

The source file is a Jira-style export with these raw fields per issue:

| Raw field | Meaning |
|---|---|
| `Issue key` | Human-readable ticket ID, e.g. `INO-21920` — this is our primary identifier |
| `Issue id` | Internal numeric Jira ID |
| `Summary` | The only free-text field (no separate description/comments/resolution exist) |
| `Assignee`, `Assignee Id` | Who the issue is assigned to |
| `Reporter`, `Reporter Id` | Who raised the issue |
| `Status` | Real values seen: `To Do`, `In Progress`, `Done`, `Rejected`, `On Hold`, `Soft Delete` |
| `Priority` | Real values seen: `Low`, `Medium`, `High`, `Urgent` (no `Critical` tier) |
| `Updated` | Last-updated timestamp, string like `5/18/26 12:01` |
| `Due date` | Due date, string like `5/15/26 0:00`, often blank |
| `Custom field (Assigned Team)` | `Falcon Squad`, `Nova Team`, or `Summit Crew` |
| `Custom field (Story Points)` | Numeric effort estimate, often blank |

We'll normalize these into a clean internal schema (`issue_key`, `summary`,
`assigned_team`, `assignee`, `priority`, `status`, ...) that the rest of
PowerOps will build on.


## 1. Imports and setup

In [1]:
import json
import re
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

DATA_PATH = Path("../data/devops_mgmt_data.json")
DATA_PATH.exists(), DATA_PATH.resolve()


(True,
 PosixPath('/Users/svedamur/Documents/agentic-ai-simulations/devOps_rag_week2/data/devops_mgmt_data.json'))

## 2. Load the raw JSON

In [2]:
with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw_records = json.load(f)

print(f"Loaded {len(raw_records)} raw records from {DATA_PATH.name}")
print(f"Type of top-level object: {type(raw_records)}")


Loaded 1000 raw records from devops_mgmt_data.json
Type of top-level object: <class 'list'>


## 3. Schema inspection

Real-world exports are rarely 100% uniform — some records may be missing keys
that others have. We collect the *union* of all keys seen across every record,
not just `raw_records[0]`, so we don't miss a field that only appears sometimes.


In [3]:
all_keys = set()
for rec in raw_records:
    all_keys.update(rec.keys())

print(f"Union of fields across all {len(raw_records)} records ({len(all_keys)} distinct keys):")
for k in sorted(all_keys):
    print(f"  - {k}")


Union of fields across all 1000 records (13 distinct keys):
  - Assignee
  - Assignee Id
  - Custom field (Assigned Team)
  - Custom field (Story Points)
  - Due date
  - Issue id
  - Issue key
  - Priority
  - Reporter
  - Reporter Id
  - Status
  - Summary
  - Updated


In [4]:
print(json.dumps(raw_records[0], indent=2))


{
  "Issue key": "INO-21920",
  "Issue id": "7798481",
  "Summary": "SDX case  are creating but remaining in Delayed Processing Pending status -FT13",
  "Assignee": "Sullivan, Deepa (Contractor)",
  "Assignee Id": "2ca25fab6856f67786767b43",
  "Reporter": "Sullivan, Deepa (Contractor)",
  "Reporter Id": "2ca25fab6856f67786767b43",
  "Status": "In Progress",
  "Priority": "Medium",
  "Updated": "5/18/26 12:01",
  "Due date": "5/15/26 0:00",
  "Custom field (Assigned Team)": "Falcon Squad",
  "Custom field (Story Points)": "0.25"
}


## 4. Record count

In [5]:
print(f"Total raw issue records: {len(raw_records)}")


Total raw issue records: 1000


## 5. Missing / empty field checks (on raw records)

We check the fields PowerOps actually depends on. A value is treated as
"missing" if the key is absent, `None`, or an empty/whitespace-only string.


In [6]:
FIELDS_TO_CHECK = [
    "Issue key",
    "Issue id",
    "Summary",
    "Assignee",
    "Status",
    "Priority",
    "Custom field (Assigned Team)",
    "Updated",
    "Due date",
    "Custom field (Story Points)",
]

def is_missing(value) -> bool:
    return value is None or (isinstance(value, str) and value.strip() == "")

missing_counts = {}
for field in FIELDS_TO_CHECK:
    missing = sum(1 for rec in raw_records if is_missing(rec.get(field)))
    missing_counts[field] = missing

missing_df = pd.DataFrame(
    [{"field": f, "missing_count": c, "missing_pct": round(100 * c / len(raw_records), 1)}
     for f, c in missing_counts.items()]
).sort_values("missing_count", ascending=False).reset_index(drop=True)

missing_df


,field,missing_count,missing_pct
0,Custom field (Story Points),224,22.4
1,Due date,158,15.8
2,Issue id,0,0.0
3,Issue key,0,0.0
4,Summary,0,0.0
5,Assignee,0,0.0
6,Priority,0,0.0
7,Status,0,0.0
8,Updated,0,0.0
9,Custom field (Assigned Team),0,0.0


## 6. Duplicate `Issue key` checks

`Issue key` is meant to be the unique identifier for each issue. We check for
exact duplicates so downstream Pinecone upserts (keyed by `Issue key`) don't
silently overwrite unrelated issues.


In [7]:
key_counter = Counter(rec.get("Issue key") for rec in raw_records)
duplicates = {k: c for k, c in key_counter.items() if c > 1}

print(f"Distinct Issue key values: {len(key_counter)}")
print(f"Duplicate Issue key values: {len(duplicates)}")
if duplicates:
    print("Examples:", list(duplicates.items())[:10])
else:
    print("No duplicate Issue keys found.")


Distinct Issue key values: 1000
Duplicate Issue key values: 0
No duplicate Issue keys found.


## 7. Data normalization

We map the raw Jira-style fields onto a clean internal schema that the rest of
PowerOps (Notebooks 02–06, and later `src/`) will use consistently. This
insulates the rest of the pipeline from the source export's naming quirks.

Date parsing: timestamps look like `5/18/26 12:01` (`%m/%d/%y %H:%M`). Some
`Due date` values are empty strings — these become `None`. If a date string
doesn't match the expected format, we keep the raw string and set the parsed
datetime to `None` rather than crashing the pipeline.


In [8]:
def _parse_jira_datetime(value):
    """Parse a 'M/D/YY H:MM' style Jira export timestamp. Returns None if blank/unparseable."""
    if value is None or (isinstance(value, str) and value.strip() == ""):
        return None
    try:
        return datetime.strptime(value.strip(), "%m/%d/%y %H:%M")
    except ValueError:
        return None


def _clean_str(value):
    """Return a stripped string, or None if the value is missing/blank."""
    if value is None:
        return None
    s = str(value).strip()
    return s if s else None


def _parse_story_points(value):
    s = _clean_str(value)
    if s is None:
        return None
    try:
        return float(s)
    except ValueError:
        return None


def normalize_issue(raw: dict) -> dict:
    """Map one raw Jira-export record onto PowerOps' internal issue schema."""
    updated_raw = raw.get("Updated")
    due_raw = raw.get("Due date")
    return {
        "issue_key": _clean_str(raw.get("Issue key")),
        "issue_id": _clean_str(raw.get("Issue id")),
        "summary": _clean_str(raw.get("Summary")),
        "assignee": _clean_str(raw.get("Assignee")),
        "assignee_id": _clean_str(raw.get("Assignee Id")),
        "reporter": _clean_str(raw.get("Reporter")),
        "reporter_id": _clean_str(raw.get("Reporter Id")),
        "status": _clean_str(raw.get("Status")),
        "priority": _clean_str(raw.get("Priority")),
        "assigned_team": _clean_str(raw.get("Custom field (Assigned Team)")),
        "story_points": _parse_story_points(raw.get("Custom field (Story Points)")),
        "updated_raw": _clean_str(updated_raw),
        "updated_dt": _parse_jira_datetime(updated_raw),
        "due_date_raw": _clean_str(due_raw),
        "due_date_dt": _parse_jira_datetime(due_raw),
    }


normalized_issues = [normalize_issue(rec) for rec in raw_records]
print(f"Normalized {len(normalized_issues)} issues.")
print(json.dumps(normalized_issues[0], indent=2, default=str))


Normalized 1000 issues.
{
  "issue_key": "INO-21920",
  "issue_id": "7798481",
  "summary": "SDX case  are creating but remaining in Delayed Processing Pending status -FT13",
  "assignee": "Sullivan, Deepa (Contractor)",
  "assignee_id": "2ca25fab6856f67786767b43",
  "reporter": "Sullivan, Deepa (Contractor)",
  "reporter_id": "2ca25fab6856f67786767b43",
  "status": "In Progress",
  "priority": "Medium",
  "assigned_team": "Falcon Squad",
  "story_points": 0.25,
  "updated_raw": "5/18/26 12:01",
  "updated_dt": "2026-05-18 12:01:00",
  "due_date_raw": "5/15/26 0:00",
  "due_date_dt": "2026-05-15 00:00:00"
}


## 8. Validation function

PowerOps must never index or answer from an incomplete issue. `validate_issue`
checks the mandatory fields every downstream stage (document building,
metadata filtering, citations) depends on.


In [9]:
MANDATORY_FIELDS = ["issue_key", "summary", "assigned_team", "priority", "status"]

def validate_issue(issue: dict) -> tuple[bool, list[str]]:
    """Validate a normalized issue dict.

    Returns (is_valid, problems) where problems is a list of human-readable
    validation failures (empty list if valid).
    """
    problems = []
    for field in MANDATORY_FIELDS:
        if not issue.get(field):
            problems.append(f"missing or empty required field: '{field}'")
    return (len(problems) == 0, problems)


# Quick smoke test
_ok, _problems = validate_issue({"issue_key": "INO-1", "summary": "x", "assigned_team": "Falcon Squad",
                                  "priority": "High", "status": "Open"})
assert _ok and _problems == []
_ok, _problems = validate_issue({"issue_key": "", "summary": None, "assigned_team": "Falcon Squad",
                                  "priority": "High", "status": "Open"})
assert not _ok and len(_problems) == 2
print("validate_issue() smoke tests passed.")


validate_issue() smoke tests passed.


## 9. Run validation across the full dataset

In [10]:
valid_issues = []
invalid_issues = []  # list of (issue, problems)

for issue in normalized_issues:
    ok, problems = validate_issue(issue)
    if ok:
        valid_issues.append(issue)
    else:
        invalid_issues.append((issue, problems))

print(f"Valid issues:   {len(valid_issues)}")
print(f"Invalid issues: {len(invalid_issues)}")


Valid issues:   1000
Invalid issues: 0


## 10. Build a Pandas DataFrame for exploration

In [11]:
df = pd.DataFrame(valid_issues)
print(df.shape)
df.head(5)


(1000, 15)


,issue_key,issue_id,summary,assignee,assignee_id,reporter,reporter_id,status,priority,assigned_team,story_points,updated_raw,updated_dt,due_date_raw,due_date_dt
0,INO-21920,7798481,SDX case are creating but remaining in Delaye...,"Sullivan, Deepa (Contractor)",2ca25fab6856f67786767b43,"Sullivan, Deepa (Contractor)",2ca25fab6856f67786767b43,In Progress,Medium,Falcon Squad,0.25,5/18/26 12:01,2026-05-18 12:01:00,5/15/26 0:00,2026-05-15
1,INO-21914,7798264,SMK4-EB26Q2R4.0.0-Beta-05/13/26-#19-7d42a03,"Mason, Ashley (Contractor)",0f46cfdfdef5207918795ef3,"Sengupta, Kathleen",751361:6a207d27-65a9-230f-d0a8-73f7a1a72e3e,Done,Medium,Falcon Squad,NaN,6/1/26 12:24,2026-06-01 12:24:00,5/13/26 0:00,2026-05-13
2,INO-21903,7798172,Add additional meta-data in DynamoDB and Updat...,"Graham, Laura",747652:5d07919e-4fab-6317-1649-646b84d288c5,"Wells, Jack (Contractor)",515f29bd07633b7e681634ff,Done,Medium,Falcon Squad,2.00,5/20/26 13:02,2026-05-20 13:02:00,5/22/26 0:00,2026-05-22
3,INO-21899,7798082,James Narayan DR - Pre Validation steps,"Mason, Anthony",a064c2957cac42b13d72aca0,"Mason, Anthony",a064c2957cac42b13d72aca0,Done,Medium,Falcon Squad,1.00,5/15/26 16:25,2026-05-15 16:25:00,5/15/26 0:00,2026-05-15
4,INO-21897,7798065,Tosca Database ia0ml005 and ia0ml006 Patch and...,"Kennedy, Jason",bcd67d3a3e34fb912af3c9e9,"Kennedy, Jason",bcd67d3a3e34fb912af3c9e9,Done,High,Falcon Squad,3.00,5/21/26 6:24,2026-05-21 06:24:00,5/20/26 0:00,2026-05-20


In [12]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 15 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   issue_key      1000 non-null   str           
 1   issue_id       1000 non-null   str           
 2   summary        1000 non-null   str           
 3   assignee       1000 non-null   str           
 4   assignee_id    1000 non-null   str           
 5   reporter       1000 non-null   str           
 6   reporter_id    1000 non-null   str           
 7   status         1000 non-null   str           
 8   priority       1000 non-null   str           
 9   assigned_team  1000 non-null   str           
 10  story_points   776 non-null    float64       
 11  updated_raw    1000 non-null   str           
 12  updated_dt     1000 non-null   datetime64[us]
 13  due_date_raw   842 non-null    str           
 14  due_date_dt    842 non-null    datetime64[us]
dtypes: datetime64[us](2), float64(1),

## 11. Distribution by `assigned_team`

In [13]:
team_counts = df["assigned_team"].value_counts()
team_counts


assigned_team
Nova Team       438
Falcon Squad    365
Summit Crew     197
Name: count, dtype: int64

## 12. Distribution by `assignee` (top 15)

In [14]:
assignee_counts = df["assignee"].value_counts()
print(f"Distinct assignees: {assignee_counts.shape[0]}")
assignee_counts.head(15)


Distinct assignees: 62


assignee
Mason, Ashley (Contractor)      181
Gibson, Anjali                  125
Rice, Anjali (Contractor)       102
Shaw, Elizabeth                  86
Myers, Deepa                     84
Mason, Anthony                   35
Mason, Kevin (Contractor)        32
Reyes, Cynthia                   29
Kennedy, Jason                   28
Bose, Jacob (Contractor)         28
Bose, Alexander                  28
Webb, Amy                        28
Wells, Jack (Contractor)         22
Prasad, Carolyn (Contractor)     20
Porter, Anjali (Contractor)      19
Name: count, dtype: int64

## 13. Distribution by `priority`

In [15]:
priority_counts = df["priority"].value_counts()
priority_counts


priority
Medium    879
High       91
Urgent     27
Low         3
Name: count, dtype: int64

## 14. Distribution by `status`

In [16]:
status_counts = df["status"].value_counts()
status_counts


status
Done           888
Soft Delete     53
Rejected        22
To Do           22
In Progress     13
On Hold          2
Name: count, dtype: int64

## 15. Cross-tab: team × status

A quick sanity view of how issues are distributed across teams and statuses —
useful for spotting whether e.g. every team actually has issues in every
status (relevant later for testing the metadata-filtered retriever).


In [17]:
pd.crosstab(df["assigned_team"], df["status"])


status,Done,In Progress,On Hold,Rejected,Soft Delete,To Do
assigned_team,,,,,,
Falcon Squad,346,2,0,9,3,5
Nova Team,364,6,2,9,49,8
Summit Crew,178,5,0,4,1,9


## 16. Vocabulary extraction

Notebook 04 will implement a **deterministic** query parser that recognizes
structured terms in natural-language questions (team names, assignee names,
priority levels, status values). Rather than hardcoding guesses, that parser
should be built from the *actual* distinct values present in this dataset —
extracted here and persisted for reuse.


In [18]:
vocabulary = {
    "assigned_teams": sorted(df["assigned_team"].dropna().unique().tolist()),
    "priorities": sorted(df["priority"].dropna().unique().tolist()),
    "statuses": sorted(df["status"].dropna().unique().tolist()),
    "assignees": sorted(df["assignee"].dropna().unique().tolist()),
}

print("Teams:     ", vocabulary["assigned_teams"])
print("Priorities:", vocabulary["priorities"])
print("Statuses:  ", vocabulary["statuses"])
print(f"Distinct assignees: {len(vocabulary['assignees'])} (showing first 10): {vocabulary['assignees'][:10]}")


Teams:      ['Falcon Squad', 'Nova Team', 'Summit Crew']
Priorities: ['High', 'Low', 'Medium', 'Urgent']
Statuses:   ['Done', 'In Progress', 'On Hold', 'Rejected', 'Soft Delete', 'To Do']
Distinct assignees: 62 (showing first 10): ['Banerjee, Ashley (Contractor)', 'Barnes, Ashley', 'Barnes, Gregory', 'Bennett, Debra', 'Bhat, Helen', 'Bhat, Maria (Contractor)', 'Bose, Alexander', 'Bose, Jacob (Contractor)', 'Chatterjee, Ashley', 'Chawla, Gregory']


In [19]:
VOCAB_PATH = Path("../data/vocabulary.json")
with open(VOCAB_PATH, "w", encoding="utf-8") as f:
    json.dump(vocabulary, f, indent=2)

print(f"Saved vocabulary to {VOCAB_PATH.resolve()}")


Saved vocabulary to /Users/svedamur/Documents/agentic-ai-simulations/devOps_rag_week2/data/vocabulary.json


## 17. Example normalized records

In [20]:
for issue in valid_issues[:3]:
    print(json.dumps(issue, indent=2, default=str))
    print("-" * 60)


{
  "issue_key": "INO-21920",
  "issue_id": "7798481",
  "summary": "SDX case  are creating but remaining in Delayed Processing Pending status -FT13",
  "assignee": "Sullivan, Deepa (Contractor)",
  "assignee_id": "2ca25fab6856f67786767b43",
  "reporter": "Sullivan, Deepa (Contractor)",
  "reporter_id": "2ca25fab6856f67786767b43",
  "status": "In Progress",
  "priority": "Medium",
  "assigned_team": "Falcon Squad",
  "story_points": 0.25,
  "updated_raw": "5/18/26 12:01",
  "updated_dt": "2026-05-18 12:01:00",
  "due_date_raw": "5/15/26 0:00",
  "due_date_dt": "2026-05-15 00:00:00"
}
------------------------------------------------------------
{
  "issue_key": "INO-21914",
  "issue_id": "7798264",
  "summary": "SMK4-EB26Q2R4.0.0-Beta-05/13/26-#19-7d42a03",
  "assignee": "Mason, Ashley (Contractor)",
  "assignee_id": "0f46cfdfdef5207918795ef3",
  "reporter": "Sengupta, Kathleen",
  "reporter_id": "751361:6a207d27-65a9-230f-d0a8-73f7a1a72e3e",
  "status": "Done",
  "priority": "Medium",


## 18. Persist normalized, validated issues

Downstream notebooks (starting with Notebook 02, Document preparation) should
load from this cleaned file rather than re-deriving normalization logic, so
there is a single source of truth for "what counts as a valid PowerOps issue."


In [21]:
OUTPUT_PATH = Path("../data/normalized_issues.json")

def _default(o):
    if isinstance(o, datetime):
        return o.isoformat()
    raise TypeError(f"Object of type {type(o)} is not JSON serializable")

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(valid_issues, f, indent=2, default=_default)

print(f"Saved {len(valid_issues)} validated issues to {OUTPUT_PATH.resolve()}")


Saved 1000 validated issues to /Users/svedamur/Documents/agentic-ai-simulations/devOps_rag_week2/data/normalized_issues.json


## 19. Validation summary

In [22]:
print("=" * 60)
print("VALIDATION SUMMARY")
print("=" * 60)
print(f"Total raw records:        {len(raw_records)}")
print(f"Valid records:            {len(valid_issues)}")
print(f"Invalid records:          {len(invalid_issues)}")
print()

if invalid_issues:
    print("Sample validation problems (first 10):")
    for issue, problems in invalid_issues[:10]:
        key = issue.get("issue_key") or "(no issue_key)"
        print(f"  - {key}: {problems}")
else:
    print("No validation problems found — every record has all mandatory fields.")


VALIDATION SUMMARY
Total raw records:        1000
Valid records:            1000
Invalid records:          0

No validation problems found — every record has all mandatory fields.


## Summary & next steps

- Loaded the raw DevOps export (see counts above), spanning three teams:
  `Falcon Squad`, `Nova Team`, `Summit Crew`.
- Normalized raw Jira-style field names into a clean internal schema
  (`issue_key`, `summary`, `assigned_team`, `assignee`, `priority`, `status`,
  `updated_dt`, `due_date_dt`, `story_points`, ...).
- Validated every record against the mandatory PowerOps fields and saved
  **`data/normalized_issues.json`** — the authoritative cleaned dataset for
  the rest of the pipeline.
- Extracted and saved **`data/vocabulary.json`** — the real teams, priorities,
  statuses, and assignees, which Notebook 04's deterministic filter parser
  will use instead of hardcoded guesses.

**Next: Notebook 02 — Document Preparation.** We'll convert each validated
issue into a LangChain `Document` (Summary + rendered metadata as
`page_content`, structured fields preserved as `metadata`) and decide on a
chunking strategy.
